# 人类活动识别模型训练

## 1. 克隆项目代码

In [ ]:
# 克隆项目代码
!git clone https://github.com/sannaraek/Lightweight-Transformer-Models-For-HAR-on-Mobile-Devices.git
%cd Lightweight-Transformer-Models-For-HAR-on-Mobile-Devices

## 2. 安装依赖

In [ ]:
# 安装依赖
!pip install -r requirements.txt
!pip install hickle requests

In [ ]:
# 导入必要的库
import os
import json
import numpy as np
import tensorflow as tf
import argparse
import time
from sklearn.utils import class_weight
import hickle as hkl

# 设置随机种子
SEED = 1
os.environ['PYTHONHASHSEED']=str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# 配置参数
DEFAULT_CONFIG = {
    "dataset": "MotionSense",
    "batch_size": 64,
    "epochs": 200,
    "learning_rate": 5e-3,
    "dropout_rate": 0.3,
    "segment_size": 128,
    "num_channels": 6,
}

# 活动标签映射
ACTIVITY_LABELS = {
    'MotionSense': ['Downstairs', 'Upstairs', 'Sitting', 'Standing', 'Walking', 'Jogging'],
    'UCI': ['Walking', 'Upstair','Downstair', 'Sitting', 'Standing', 'Lying'],
    'HHAR': ['Sitting', 'Standing', 'Walking', 'Upstair', 'Downstairs', 'Biking'],
    'RealWorld': ['Downstairs','Upstairs', 'Jumping','Lying', 'Running', 'Sitting', 'Standing', 'Walking'],
    'SHL': ['Standing','Walking','Runing','Biking','Car','Bus','Train','Subway']
}

# 数据集配置
DATASET_CONFIG = {
    'MotionSense': {
        'num_users': 24,
        'num_classes': 6
    },
    'UCI': {
        'num_users': None,
        'num_classes': 6
    }
}

In [ ]:
def load_dataset(dataset_name):
    """加载数据集"""
    if dataset_name == 'MotionSense':
        data_path = 'datasetStandardized/MotionSense'
        
        all_data = []
        all_labels = []
        num_users = DATASET_CONFIG[dataset_name]['num_users']
        for i in range(num_users):
            user_data = hkl.load(f'{data_path}/UserData{i}.hkl')
            user_label = hkl.load(f'{data_path}/UserLabel{i}.hkl')
            all_data.append(user_data)
            all_labels.append(user_label)
        
        X = np.vstack(all_data)
        y = np.concatenate(all_labels)
        
        y = tf.keras.utils.to_categorical(y, num_classes=6)
        
        X_train = X[:int(0.8 * len(X))]
        X_test = X[int(0.8 * len(X)):]
        y_train = y[:int(0.8 * len(y))]
        y_test = y[int(0.8 * len(y)):]
        
    elif dataset_name == 'UCI':
        from datasets.DATA_UCI import load_data
        X_train, y_train, X_test, y_test = load_data()
    else:
        raise ValueError(f"Dataset {dataset_name} not implemented yet")
    
    return X_train, y_train, X_test, y_test

In [ ]:
def preprocess_data(X_train, X_test):
    """数据预处理"""
    mean_acc = np.mean(X_train[:,:,:3])
    std_acc = np.std(X_train[:,:,:3])
    X_train[:,:,:3] = (X_train[:,:,:3] - mean_acc) / std_acc
    X_test[:,:,:3] = (X_test[:,:,:3] - mean_acc) / std_acc
    
    mean_gyro = np.mean(X_train[:,:,3:])
    std_gyro = np.std(X_train[:,:,3:])
    X_train[:,:,3:] = (X_train[:,:,3:] - mean_gyro) / std_gyro
    X_test[:,:,3:] = (X_test[:,:,3:] - mean_gyro) / std_gyro
    
    return X_train, X_test, {
        'mean_acc': mean_acc, 'std_acc': std_acc,
        'mean_gyro': mean_gyro, 'std_gyro': std_gyro
    }

In [ ]:
def create_mobile_model(input_shape, num_classes):
    """创建轻量级MobileHART模型"""
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    acc_input = tf.keras.layers.Lambda(lambda x: x[:,:,:3])(inputs)
    gyro_input = tf.keras.layers.Lambda(lambda x: x[:,:,3:])(inputs)
    
    base_model = model.HART(
        input_shape=input_shape,
        activityCount=num_classes,
        projection_dim=32,
        patchSize=16,
        timeStep=16,
        num_heads=2,
        filterAttentionHead=2,
        convKernels=[3, 7, 15],
        mlp_head_units=[256],
        dropout_rate=DEFAULT_CONFIG['dropout_rate']
    )
    
    return base_model

In [ ]:
def convert_to_tflite(model, dataset_name, preprocessing_params):
    """转换为TFLite模型"""
    os.makedirs('mobile_models', exist_ok=True)
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    
    model_path = f'mobile_models/mobile_hart_{dataset_name.lower()}.tflite'
    with open(model_path, 'wb') as f:
        f.write(tflite_model)
    
    params_dict = {
        'mean_acc': float(preprocessing_params['mean_acc']),
        'std_acc': float(preprocessing_params['std_acc']),
        'mean_gyro': float(preprocessing_params['mean_gyro']),
        'std_gyro': float(preprocessing_params['std_gyro'])
    }
    
    params_path = f'mobile_models/preprocessing_params_{dataset_name.lower()}.json'
    with open(params_path, 'w', encoding='utf-8') as f:
        json.dump(params_dict, f, indent=2)
    
    print(f"Model saved to {model_path}")
    print(f"Preprocessing parameters saved to {params_path}")

## 2. 开始训练

In [ ]:
# 设置训练参数
args = type('Args', (), {
    'dataset': DEFAULT_CONFIG['dataset'],
    'batch_size': DEFAULT_CONFIG['batch_size'],
    'epochs': DEFAULT_CONFIG['epochs'],
    'learning_rate': DEFAULT_CONFIG['learning_rate']
})()

# 加载数据集
print(f"Loading {args.dataset} dataset...")
X_train, y_train, X_test, y_test = load_dataset(args.dataset)

# 数据预处理
print("Preprocessing data...")
X_train, X_test, preprocessing_params = preprocess_data(X_train, X_test)

# 获取输入形状和类别数
input_shape = (DEFAULT_CONFIG['segment_size'], DEFAULT_CONFIG['num_channels'])
num_classes = len(ACTIVITY_LABELS[args.dataset])

# 创建模型
print("Creating MobileHART model...")
model = create_mobile_model(input_shape, num_classes)

# 编译模型
model.compile(
    optimizer=tf.keras.optimizers.Adam(args.learning_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

# 设置检查点
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(
    checkpoint_dir, 
    f"mobile_hart_{args.dataset.lower()}.weights.h5"
)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# 计算类别权重
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(np.argmax(y_train, axis=1)),
    y=np.argmax(y_train, axis=1)
)
class_weights = dict(enumerate(class_weights))

# 训练模型
print("Training model...")
history = model.fit(
    X_train, y_train,
    batch_size=args.batch_size,
    epochs=args.epochs,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[checkpoint_callback],
    verbose=1
)

# 评估模型
print("Evaluating model...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_accuracy*100:.2f}%")

# 转换为TFLite
print("Converting to TFLite...")
convert_to_tflite(model, args.dataset, preprocessing_params)

## 3. 保存结果到 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 创建结果目录
results_dir = '/content/drive/MyDrive/har_results'
!mkdir -p {results_dir}

# 复制模型和参数文件
!cp -r mobile_models/* {results_dir}/
!cp -r checkpoints/* {results_dir}/